# exp489 acceleration-state fixed32 mechanism inference guard

exp489 is implemented only as a train-side Stage 0B mechanism preflight.
Hidden-test regeneration, submission creation, and inference remain disabled.

## Contents

1. Imports and notebook-safe configuration
2. Fail-closed inference contract
3. Guarded orchestration

## 1. Imports and notebook-safe configuration

In [ ]:
from __future__ import annotations

import json
from collections.abc import Mapping
from pathlib import Path
from typing import Any

import yaml

EXPERIMENT_NAME = "exp489_acceleration_state_fixed32_mechanism_audit"
PACKAGE_DIR = Path.cwd()


def get_nested(mapping: Mapping[str, Any], dotted_key: str, default: Any = None) -> Any:
    value: Any = mapping
    for part in dotted_key.split("."):
        if not isinstance(value, Mapping) or part not in value:
            return default
        value = value[part]
    return value


def find_project_root(start: Path = PACKAGE_DIR) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "project.yml").is_file():
            return candidate
    return start


def config_path() -> Path:
    root = find_project_root()
    for candidate in (
        PACKAGE_DIR / "config.yaml",
        root / "experiments" / EXPERIMENT_NAME / "config.yaml",
    ):
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("exp489 config.yaml was not found")


def load_config(path: Path | None = None) -> dict[str, Any]:
    resolved = config_path() if path is None else path
    value = yaml.safe_load(resolved.read_text()) or {}
    if not isinstance(value, dict):
        raise ValueError(f"{resolved} must contain a YAML mapping")
    return value

## 2. Fail-closed inference contract

In [ ]:
def validate_inference_disabled(config: Mapping[str, Any]) -> dict[str, bool]:
    if get_nested(config, "experiment.name") != EXPERIMENT_NAME:
        raise ValueError("wrong exp489 config")
    contract = {
        "implementation_authorized": bool(
            get_nested(config, "execution.implementation_authorized", False)
        ),
        "canonical_notebook_adoption_authorized": bool(
            get_nested(
                config,
                "execution.canonical_notebook_adoption_authorized",
                False,
            )
        ),
        "kaggle_package_authorized": bool(
            get_nested(config, "execution.kaggle_package_authorized", False)
        ),
        "stage0a_run_authorized": bool(
            get_nested(config, "execution.stage0a_run_authorized", False)
        ),
        "stage0b_run_authorized": bool(
            get_nested(config, "execution.stage0b_run_authorized", True)
        ),
        "stage1_run_authorized": bool(
            get_nested(config, "execution.stage1_run_authorized", True)
        ),
        "inference_authorized": bool(
            get_nested(config, "execution.inference_authorized", True)
        ),
        "submission_authorized": bool(
            get_nested(config, "execution.submission_authorized", True)
        ),
        "create_submission": bool(
            get_nested(config, "execution.create_submission", True)
        ),
    }
    if not contract["implementation_authorized"]:
        raise RuntimeError("exp489 implementation is not authorized")
    forbidden = {
        key: contract[key]
        for key in (
            "inference_authorized",
            "submission_authorized",
            "create_submission",
        )
        if contract[key]
    }
    if forbidden:
        raise ValueError(f"exp489 inference contract was unlocked: {forbidden}")
    if get_nested(config, "inference.mode") not in {
        "disabled_stage0b_preflight",
        "disabled_stage0b_fail_closed",
    }:
        raise ValueError("exp489 inference mode changed")
    return contract


def run_inference(config: Mapping[str, Any]) -> None:
    validate_inference_disabled(config)
    raise RuntimeError(
        "exp489 inference is disabled until all train-side gates pass and a "
        "later, separate inference authorization is recorded."
    )

## 3. Guarded orchestration

In [ ]:
if __name__ == "__main__":
    CONFIG = load_config()
    CONTRACT = validate_inference_disabled(CONFIG)
    print(
        json.dumps(
            {
                "event": "exp489_inference_disabled",
                "experiment": EXPERIMENT_NAME,
                "status": get_nested(CONFIG, "experiment.status"),
                "inference_contract": CONTRACT,
                "message": "Inference and submission remain fail-closed.",
            },
            indent=2,
            sort_keys=True,
        )
    )
    run_inference(CONFIG)